In [13]:
# You may need to run: !pip install pandas openpyxl
import re
import ollama
import pandas as pd
from tqdm.notebook import tqdm

# Define your local Ollama models
MODELS = ["gemma4:e4b"]
# NOTE: double-check that the model names below actually exist in your local Ollama
# install. Run `ollama list` in a terminal — if a name in MODELS isn't there, the
# whole evaluation for that model will fail silently. In particular, "*-mlx" tags
# come from Apple's MLX framework and are NOT standard Ollama tags. If you have
# an MLX model, replace it with the actual Ollama tag (e.g. "gemma3:27b" or
# whatever `ollama list` shows).


# Load the questions from the Excel file
print("Loading Sorular.xlsx...")
df = pd.read_excel("Sorular.xlsx")
print(f"Successfully loaded {len(df)} questions from the dataset.")

Loading Sorular.xlsx...
Successfully loaded 100 questions from the dataset.


In [15]:
df

,ID,TYPE,PROBLEM (TURKISH),PROBLEM (ENGLISH),SOLUTION (TURKISH),SOLUTION (ENGLISH),ANSWER,notes/questions
0,1,1,Bir havalimanında belirlenen bir günde yapılan...,"At an airport, one-fourth of the total number ...","60 uçuş tüm uçuş sayısının 5'te 3'ü ise, tüm u...",Since 60 flights represent three-fifths of the...,75,2025 msü sınavından alınmış sayıları değiştiri...
1,2,2,Bir havalimanında belirlenen bir günde yapılan...,"At an airport, a/b of the total number of flig...",x tane uçuş tüm uçuş sayısının c/d'sine eşit i...,If x flights correspond to c/d of the total nu...,x(d/c)*(b - a)/b,Sorunun sonunda verilen koşullar modellerin ha...
2,3,3,Bir havalimanında belirlenen bir günde yapılan...,"At an airport, one-fourth of the total number ...","""Yurt içi uçuşların her birinde 5 kabin memuru...","The expression ""5 cabin crew members served on...",75,"Sorunun dikkat dağıtıcı cümlesi ""Yurt içi uçu..."
3,4,4,Bir havalimanında belirlenen bir günde yapılan...,"At an airport, one-fourth of the total number ...","70 uçuş, toplam uçuş sayısının 3/5'ine eşitse,...",If 70 flights correspond to 3/5 of the total n...,çözüm yok.,ill-posed yapı: uçuş sayısının tam sayı çıkmam...
4,5,1,Bir araç A şehrinden B şehrine 100 km'lik yolu...,A vehicle traveled a 100 km distance from city...,Problemin çözümüne başlamadan önce ortalama hı...,"Before starting the solution to the problem, l...",1900-04-09 00:00:00,NaN
...,...,...,...,...,...,...,...,...
95,96,4,"Bir yaz kampına katılan öğrenciler, her çadırd...",Students attending a summer camp are placed in...,12 çadırda her çadırda x öğrenci olsun.\nTopla...,Let x be the number of students in each tent w...,çözüm yok.,ill-posed yapı: çadırdaki öğrenci sayılarının ...
96,97,1,Bir okulda düzenlenen bilgi yarışmasına 10 tak...,A quiz competition is organized at a school wi...,10 takımın yaptığı toplam karşılaşma sayısı:\n...,Total number of matches played by 10 teams:\n1...,18,2017 YGS sorusunun sayıları ve senaryosu değiş...
97,98,2,Bir okulda düzenlenen bilgi yarışmasına n sayı...,A quiz competition is organized at a school wi...,n sayıda takımın yaptığı toplam karşılaşma say...,Total number of matches played by 10 teams:\nn...,y.n.(n-1) / 2x,NaN
98,99,3,Bir okulda düzenlenen bilgi yarışmasına 10 tak...,A quiz competition is organized at a school wi...,Jüri üyelerinin aldığı ücret ve yarışmanın kaç...,The information regarding the payment and the ...,18,Jüri üyelerinin aldığı ücret ve yarışmanın kaç...


In [17]:
# Hardcoded 8-shot prompt built from the 8 example problems (English versions).
# Examples 1, 2, 3, 5, 6, 7 are solvable; examples 4 and 8 are mathematically
# impossible. For the solvable ones we end with "#### <number>" so the model
# learns the final-answer format expected by extract_final_answer().
# For impossible ones we end with "#### IMPOSSIBLE".

SHOTS = [
    # ---------------- Shot 1 ----------------
    ("A grocer buys 4 oranges for 60 TL and sells 6 oranges for 120 TL. "
     "How many oranges must the grocer sell to make a profit of 420 TL?",
     "Cost of 4 oranges = 60 TL, so cost per orange = 60 / 4 = 15 TL.\n"
     "Selling price of 6 oranges = 120 TL, so selling price per orange = 120 / 6 = 20 TL.\n"
     "Profit per orange = 20 - 15 = 5 TL.\n"
     "Number of oranges needed = 420 / 5 = 84.\n"
     "#### 84"),

    # ---------------- Shot 2 (symbolic) ----------------
    ("A grocer buys x oranges for A TL and sells y oranges for B TL. "
     "How many oranges must the grocer sell to make a profit of C TL? "
     "Find the answer in terms of A, B, C, x and y.",
     "Cost per orange = A / x.\n"
     "Selling price per orange = B / y.\n"
     "Profit per orange = (B / y) - (A / x).\n"
     "Number of oranges sold = C / [(B / y) - (A / x)].\n"
     "#### C / ((B/y) - (A/x))"),

    # ---------------- Shot 3 (with distractor) ----------------
    ("A grocer buys 4 oranges for 60 TL and sells 6 oranges for 120 TL. "
     "One third of the oranges are slightly smaller than average. "
     "How many oranges must the grocer sell to make a profit of 420 TL?",
     "The statement about one third of the oranges being smaller than average "
     "is a distractor and does not affect the calculation.\n"
     "Cost per orange = 60 / 4 = 15 TL.\n"
     "Selling price per orange = 120 / 6 = 20 TL.\n"
     "Profit per orange = 20 - 15 = 5 TL.\n"
     "Number of oranges needed = 420 / 5 = 84.\n"
     "#### 84"),

    # ---------------- Shot 4 (impossible) ----------------
    ("A grocer buys 4 oranges for 60 TL and sells 7 oranges for 70 TL. "
     "How many oranges must the grocer sell to make a profit of 420 TL? "
     "Find the answer as a positive integer.",
     "Cost per orange = 60 / 4 = 15 TL.\n"
     "Selling price per orange = 70 / 7 = 10 TL.\n"
     "Profit per orange = 10 - 15 = -5 TL, which is a loss.\n"
     "Since the per-orange profit is negative, a profit of 420 TL cannot be achieved. "
     "The question is mathematically not correct.\n"
     "#### IMPOSSIBLE"),

    # ---------------- Shot 5 ----------------
    ("In a company, the number of female colleagues of each female employee is "
     "4 more than twice the number of her male colleagues. For each male employee, "
     "the number of his female colleagues is 2 less than three times the number of "
     "his male colleagues. How many employees are there in the company in total?",
     "Let K = number of women, E = number of men.\n"
     "For each woman: other women = K - 1, men = E, so K - 1 = 2E + 4, giving K = 2E + 5.\n"
     "For each man: women = K, other men = E - 1, so K = 3(E - 1) - 2 = 3E - 5.\n"
     "Equating: 2E + 5 = 3E - 5, so E = 10 and K = 25.\n"
     "Total = K + E = 25 + 10 = 35.\n"
     "#### 35"),

    # ---------------- Shot 6 (symbolic) ----------------
    ("In a company, for each female employee the number of female colleagues is x times "
     "the number of male colleagues. For each male employee the number of female colleagues "
     "is y times the number of male colleagues. Find the total number of employees in "
     "terms of x and y.",
     "Let K = women, E = men.\n"
     "For each woman: K - 1 = xE, so K = xE + 1.\n"
     "For each man: K = y(E - 1) = yE - y.\n"
     "Equating: xE + 1 = yE - y, so (y - x)E = y + 1 and E = (y + 1) / (y - x).\n"
     "Total = K + E = (x + 1)E + 1 = ((x + 1)(y + 1)) / (y - x) + 1.\n"
     "#### ((x+1)(y+1))/(y-x) + 1"),

    # ---------------- Shot 7 (with distractor) ----------------
    ("In a company, for each female employee, the number of female colleagues is "
     "4 more than twice the number of male colleagues. For each male employee, the "
     "number of female colleagues is 2 less than three times the number of male "
     "colleagues. In addition, the company has 7 departments, and each department has "
     "the same number of employees. How many employees are there in the company in total?",
     "The information about 7 departments with equal numbers of employees is a distractor "
     "and is not needed.\n"
     "Let K = women, E = men.\n"
     "For women: K - 1 = 2E + 4, so K = 2E + 5.\n"
     "For men: K = 3(E - 1) - 2 = 3E - 5.\n"
     "Equating: 2E + 5 = 3E - 5, so E = 10 and K = 25.\n"
     "Total = 25 + 10 = 35.\n"
     "#### 35"),

    # ---------------- Shot 8 (impossible) ----------------
    ("In a company, for each female employee the number of female colleagues is "
     "2 more than 5 times the number of male colleagues. For each male employee the "
     "number of female colleagues is 7 less than twice the number of male colleagues. "
     "How many employees are there in the company in total? "
     "Find the answer as a positive integer.",
     "Let K = women, E = men.\n"
     "For women: K - 1 = 5E + 2, so K = 5E + 3.\n"
     "For men: K = 2(E - 1) - 7 = 2E - 9.\n"
     "Equating: 5E + 3 = 2E - 9, so 3E = -12 and E = -4.\n"
     "A negative number of men is impossible in real life, "
     "so the question is mathematically not correct.\n"
     "#### IMPOSSIBLE"),
]

# Build the prompt string. Each example ends its answer with "#### <value>",
# so the model learns to do the same on the target question.
CUSTOM_8SHOT_PROMPT = (
    "Solve the following math word problems. Show your reasoning step by step "
    "and finish with a line of the form \"#### <final answer>\". "
    "If the problem is mathematically impossible, write \"#### IMPOSSIBLE\".\n\n"
    + "\n\n".join(f"Q: {q}\nA: {a}" for q, a in SHOTS)
    + "\n\nQ: {question}\nA:"
)

print("Your generated 8-shot template looks like this:\n")
print(CUSTOM_8SHOT_PROMPT)

Your generated 8-shot template looks like this:

Solve the following math word problems. Show your reasoning step by step and finish with a line of the form "#### <final answer>". If the problem is mathematically impossible, write "#### IMPOSSIBLE".

Q: A grocer buys 4 oranges for 60 TL and sells 6 oranges for 120 TL. How many oranges must the grocer sell to make a profit of 420 TL?
A: Cost of 4 oranges = 60 TL, so cost per orange = 60 / 4 = 15 TL.
Selling price of 6 oranges = 120 TL, so selling price per orange = 120 / 6 = 20 TL.
Profit per orange = 20 - 15 = 5 TL.
Number of oranges needed = 420 / 5 = 84.
#### 84

Q: A grocer buys x oranges for A TL and sells y oranges for B TL. How many oranges must the grocer sell to make a profit of C TL? Find the answer in terms of A, B, C, x and y.
A: Cost per orange = A / x.
Selling price per orange = B / y.
Profit per orange = (B / y) - (A / x).
Number of oranges sold = C / [(B / y) - (A / x)].
#### C / ((B/y) - (A/x))

Q: A grocer buys 4 orang

In [19]:
import time

def generate_response(model_name: str, question: str) -> str:
    """Queries the local Ollama instance with greedy decoding.

    Robustness fixes (vs. the previous version):
    - num_ctx is set explicitly to 8192 so the long 8-shot prompt fits in the
      context window. Ollama's default num_ctx is 2048, which silently truncates
      the prompt and often causes the model to return an empty string.
    - num_predict is raised from 1024 to 2048 so long chain-of-thought answers
      don't get cut off before they reach the "#### <answer>" line.
    - The call is wrapped in try/except with one retry. On the retry we slightly
      perturb num_predict to nudge the model out of any degenerate empty-output
      state.
    - If both attempts return empty text, we surface a sentinel string that
      includes the error so the row is never silently blank in the output Excel.
    """
    prompt = CUSTOM_8SHOT_PROMPT.format(question=question)

    base_options = {
        "temperature": 0.0,
        "num_predict": 2048,   # was 1024 — too small for 8-shot CoT
        "num_ctx": 8192,       # was implicit 2048 — too small for 8-shot prompt
    }

    last_error = None
    for attempt in range(2):
        try:
            opts = dict(base_options)
            if attempt == 1:
                # Small perturbation on retry to escape degenerate empty outputs.
                opts["num_predict"] = 1536
            response = ollama.generate(
                model=model_name,
                prompt=prompt,
                options=opts,
            )
            text = response.get("response", "") or ""
            if text.strip():
                return text
            # Empty response — fall through to retry / error sentinel.
            last_error = "empty response from ollama.generate"
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            time.sleep(0.5)  # brief backoff before retry

    # Both attempts failed — return a visible marker instead of "".
    return f"[GENERATION FAILED: {last_error}]"


def extract_final_answer(text: str) -> str:
    # (Keep your existing code here)
    match = re.search(r'####\s*(-?[\d,]+(?:\.\d+)?)', text)
    if match:
        return match.group(1).replace(',', '')
    match = re.search(r'answer is\s*(-?[\d,]+(?:\.\d+)?)', text, re.IGNORECASE)
    if match:
        return match.group(1).replace(',', '')
    match = re.search(r'\\boxed{([^}]+)}', text)
    if match:
        box_content = match.group(1)
        num_match = re.search(r'(-?[\d,]+(?:\.\d+)?)', box_content)
        if num_match:
            return num_match.group(1).replace(',', '')
    # Also recognise an "#### IMPOSSIBLE" marker from the 8-shot template.
    if re.search(r'####\s*IMPOSSIBLE', text, re.IGNORECASE):
        return "IMPOSSIBLE"
    numbers = re.findall(r'-?[\d,]+(?:\.\d+)?', text)
    if numbers:
        return numbers[-1].replace(',', '')
    return None


def extract_ground_truth(answer_text: str) -> str:
    # (Keep your existing code here)
    answer_text = str(answer_text)
    match = re.search(r'####\s*(-?[\d,]+(?:\.\d+)?)', answer_text)
    if match:
        return match.group(1).replace(',', '')
    return answer_text.strip().replace(',', '')

In [21]:
# Create new empty columns for each model's responses and extracted answers
for model in MODELS:
    df[f"{model}_Response"] = ""
    df[f"{model}_Extracted_Answer"] = ""

# Since the 8 few-shot examples are now hardcoded (not taken from the Excel file),
# we can evaluate the model on ALL questions in the dataset.
eval_indices = df.index

results = {model: 0 for model in MODELS}

for model in MODELS:
    print(f"\n{'='*40}\nEvaluating Model: {model}\n{'='*40}")

    correct_count = 0
    total_count = len(eval_indices)

    # Iterate through the dataframe using the index to update rows safely
    for idx in tqdm(eval_indices, desc=f"{model}"):
        question = df.at[idx, 'PROBLEM (ENGLISH)']       # Update if column name differs
        gold_answer_text = df.at[idx, 'SOLUTION (TURKISH)']  # Update if column name differs

        # 1. Get ground truth
        gold = extract_ground_truth(gold_answer_text)

        # 2. Get model prediction
        raw_response = generate_response(model, question)
        prediction = extract_final_answer(raw_response)

        # 3. Write data back to the dataframe
        df.at[idx, f"{model}_Response"] = raw_response
        df.at[idx, f"{model}_Extracted_Answer"] = prediction if prediction is not None else "NOT_FOUND"

        # 4. Compare
        try:
            if prediction is not None and float(prediction) == float(gold):
                correct_count += 1
        except ValueError:
            # Non-numeric comparison fallback (e.g., "IMPOSSIBLE")
            if prediction is not None and str(prediction).strip().upper() == str(gold).strip().upper():
                correct_count += 1

    accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
    results[model] = accuracy
    print(f"-> Accuracy for {model}: {accuracy:.2f}%\n")

# Save the updated DataFrame to a new Excel file
output_filename = "cevaplar.xlsx"
df.to_excel(output_filename, index=False)
print(f"✅ All responses and extracted answers successfully saved to '{output_filename}'!")

# Final Output Summary
print("\n" + "="*40 + "\nFINAL ACCURACIES (All Excel Questions)\n" + "="*40)
for model, acc in results.items():
    print(f"Model: {model:<15} | Accuracy: {acc:.2f}%")


Evaluating Model: gemma4:e4b


gemma4:e4b:   0%|          | 0/100 [00:00<?, ?it/s]

-> Accuracy for gemma4:e4b: 0.00%

✅ All responses and extracted answers successfully saved to 'cevaplar.xlsx'!

FINAL ACCURACIES (All Excel Questions)
Model: gemma4:e4b      | Accuracy: 0.00%
